# Дообучение своей модели для PuTTY-AI (Unsloth + Qwen2.5-7B)

**Что делает:** берёт ваш датасет `dataset.jsonl` (сделанный export_dataset.py) и дообучает Qwen2.5-7B методом LoRA. За ~30-60 минут на бесплатном Colab вы получаете GGUF-файл своей модели.

**Как пользоваться:**
1. Сначала на компьютере: `python export_dataset.py` → получите `dataset.jsonl`
2. Runtime → Change runtime type → **T4 GPU**
3. Выполните ячейки по порядку (Runtime → Run all)
4. Скачайте `my_model.gguf` в конце
5. В PuTTY-AI: Настройки ИИ → llamafile/Ollama → ваша модель

Модель останется полностью вашей: работает офлайн, знает ваши платы.

In [ ]:
# 1. Установка Unsloth (один раз, ~2 минуты)
%%capture
!pip install unsloth
!pip uninstall -y trl
!pip install --no-deps trl==0.11.4

In [ ]:
# 2. Загрузка вашего датасета
from google.colab import files
print('Выберите dataset.jsonl (сделан export_dataset.py на вашем ПК):')
uploaded = files.upload()
import os
assert 'dataset.jsonl' in os.listdir('.'), 'Файл dataset.jsonl не загружен!'

In [ ]:
# 3. Загрузка базовой модели Qwen2.5-7B (в 4-бит, экономия памяти)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-7B-Instruct',
    max_seq_length=2048,
    dtype=None,            # автоматически
    load_in_4bit=True,     # 4-бит: модель занимает ~5 ГБ VRAM
)
print('Модель загружена. VRAM, ГБ:', round(torch.cuda.memory_allocated()/1e9, 1))

In [ ]:
# 4. Добавляем LoRA-адаптер (обучаем только его — быстро и дёшево)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # ранг адаптера
    lora_alpha=16,
    lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

In [ ]:
# 5. Подготовка датасета (ShareGPT → текст с шаблоном Qwen2.5)
import json

rows = [json.loads(l) for l in open('dataset.jsonl', encoding='utf-8')]
print('Примеров в датасете:', len(rows))

def to_text(ex):
    msgs = ex['conversations']
    chat = [{'role': 'system', 'content': msgs[0]['value']}]
    for m in msgs[1:]:
        role = 'user' if m['from'] == 'human' else 'assistant'
        chat.append({'role': role, 'content': m['value']})
    return {'text': tokenizer.apply_chat_template(chat, tokenize=False,
                                                  add_generation_prompt=False)}

data = [to_text(ex) for ex in rows]
print(data[0]['text'][:400], '...')

In [ ]:
# 6. Обучение (~30-60 минут на T4 для 100-300 примеров)
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

ds = Dataset.from_list(data)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=3,        # для малого датасета 2-3 эпохи
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy='no',
        report_to='none',
    ),
)
trainer.train()
print('Обучение завершено.')

In [ ]:
# 7. Экспорт в GGUF (формат для llamafile/Ollama)
# q4_k_m — хороший баланс размера (~4.7 ГБ) и качества
model.save_pretrained_gguf('my_model', tokenizer,
                           quantization_method='q4_k_m')
print('GGUF сохранён: my_model-unsloth.Q4_K_M.gguf')

In [ ]:
# 8. Скачивание модели на компьютер
from google.colab import files
files.download('my_model-unsloth.Q4_K_M.gguf')

## Подключение к PuTTY-AI

**Вариант 1 — Ollama (проще):**
1. Создайте файл `Modelfile` (без расширения):
```
FROM my_model-unsloth.Q4_K_M.gguf
```
2. `ollama create putty-ai -f Modelfile`
3. Настройки ИИ → Ollama → модель `putty-ai`

**Вариант 2 — llamafile (1 файл, Win7):**
переименуйте gguf в любой llamafile-совместимый дистрибутив или запустите через llama-server. Укажите путь в «Настройки ИИ».

**Итерация:** чем больше работаете с PuTTY-AI, тем больше навыков → `export_dataset.py` → дообучение заново → модель умнеет с каждым циклом.